## Question

**Can embeddings represent a listener's musical taste?** A song search starts with words someone types. A recommender starts with songs someone liked and tries to find unheard songs that fit. We will test whether the titles and descriptions of liked songs give us a useful starting point.

This is a small, guided experiment. You need basic Python, but no machine learning background. For the demo, we treat three plays as *likes*; a real app would need a better signal of preference.


## Hypothesis

**Songs whose descriptions share themes with the three liked songs should rank highly among unheard songs.** We turn each song's title and description into a list of numbers called an *embedding*, then average the three liked songs' embeddings to make one search query. This average, called a *centroid*, is our stand-in for the listener's taste.

This tests whether written themes help find related songs. It cannot tell us whether the listener would enjoy how those songs sound.


## Experiment

We will use:

- **3 liked songs** from one example listener
- A **24-song fictional catalog** in `songs.csv`
- The **`all-MiniLM-L6-v2`** text embedding model
- **[Chroma](https://docs.trychroma.com/)** to store and search the embeddings
- **Cosine similarity** to rank songs by the listener's preference vector

Chroma is a vector database: it stores each song's embedding and finds songs whose vectors are close to a query. We use it because its in-memory Python client lets this small notebook run without setting up a separate database service. MiniLM creates the embeddings; Chroma stores and searches them.

The [product recommender example](https://github.com/jorshali/developers-guide-to-ai/blob/main/part3/vector_databases/product_recommender.py) inspired the history, retrieval, and filtering pattern.

```text
song title + description → embedding → Chroma
3 liked-song embeddings → average (centroid) → search → remove heard songs
```

To run along, install dependencies from the repository root with `python -m pip install -r requirements.txt`. Open this notebook in Jupyter or VS Code using that Python environment, then run its cells from top to bottom. It can load `songs.csv` when launched from either the repository root or `embeddings-and-search`. The model may download on the first run.


### Load the catalog

`Song` is a plain Python record for one CSV row. We embed each song's title and description; the genre stays separate for a later comparison.


In [1]:
import csv
from dataclasses import dataclass, field
from pathlib import Path


@dataclass(frozen=True)
class Song:
    songId: str
    title: str
    artist: str
    genre: str
    description: str

    def title_and_description(self) -> str:
        return f"{self.title} {self.description}"


song_path = Path("songs.csv")
if not song_path.exists():
    song_path = Path("embeddings-and-search") / "songs.csv"

with song_path.open(newline="", encoding="utf-8") as song_file:
    songs = [
        Song(
            songId=row["id"],
            title=row["title"],
            artist=row["artist"],
            genre=row["genre"],
            description=row["description"],
        )
        for row in csv.DictReader(song_file)
    ]

print(f"Loaded {len(songs)} songs")
print(f"Example: {songs[0].title} ({songs[0].genre}) — {songs[0].description}")


Loaded 24 songs
Example: First Light (Pop) — A song about finding hope after heartbreak.


### Embed and index the songs

MiniLM makes an embedding for each song's text. `collection.upsert` stores it under the song ID. Chroma keeps this collection in memory for the current session.

Chroma's [`space` setting](https://cookbook.chromadb.dev/core/configuration/) chooses how it compares embeddings:

| Space | What it measures |
| --- | --- |
| `cosine` | Difference in direction, regardless of vector length |
| `l2` | Squared distance between coordinates; vector length matters |
| `ip` | Inner product (dot product); direction and vector length matter |

All three use the same song embeddings but can rank them differently. We choose **cosine** because this experiment looks for descriptions pointing in a similar semantic direction. Cosine similarity reports closeness (**higher means closer**), while [Chroma reports cosine distance](https://cookbook.chromadb.dev/core/collections/) (**lower means closer**): `distance = 1 − similarity`. For example, a similarity of `0.8` is a distance of `0.2`. Both give the same ranking.


In [2]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
collection = client.get_or_create_collection(
    "song_recommendations_cosine",
    embedding_function=embedding_function,
    configuration={"hnsw": {"space": "cosine"}},
)
collection.upsert(
    ids=[song.songId for song in songs],
    documents=[song.title_and_description() for song in songs],
)
print(f"Indexed {collection.count()} songs")


Indexed 24 songs


### Record a listener's likes

These are ordinary application classes. `ListeningHistory` holds songs the person liked, and `User` provides a convenient way to add one.


In [3]:
@dataclass
class ListeningHistory:
    songs: list[Song] = field(default_factory=list)

    def add_song(self, song: Song) -> None:
        self.songs.append(song)

    def has_listened_to(self, song_id: str) -> bool:
        return any(song.songId == song_id for song in self.songs)


@dataclass
class User:
    username: str
    listening_history: ListeningHistory = field(default_factory=ListeningHistory)

    def add_listen(self, song: Song) -> None:
        self.listening_history.add_song(song)




### Build the centroid and retrieve songs

The recommender fetches the three stored embeddings, averages them, queries Chroma once, removes songs already heard, and ranks the remaining candidates. Repeated plays of a song count once. `zip(*embeddings)` groups matching coordinates for the average; `results["ids"][0]` selects the results of our one Chroma query.

The default `same_genre_weight=0` keeps the main experiment focused on cosine similarity. The optional genre preference is examined in **Analysis**. We score all 24 songs here so every unheard song can be considered. A larger catalog would retrieve a smaller candidate set first.

**Why convert Chroma's distance?** Smaller cosine distances mean closer text matches. `semantic_score = 1 - distance / 2` turns them into `0…1` scores where higher means closer, matching the genre score below. Dividing by `2` halves each distance, changing the possible range from `0…2` to `0…1`; subtracting from `1` reverses its direction.

| Chroma distance | Calculation | Semantic score |
| ---: | ---: | ---: |
| `0` | `1 - 0 / 2` | `1` (closest) |
| `0.4` | `1 - 0.4 / 2` | `0.8` |
| `2` | `1 - 2 / 2` | `0` (farthest) |

The conversion keeps the same song order; it is not a probability that someone will like a song.

**Why this final score?** Both `semantic_score` and `genre_match` run from `0` to `1`. We choose `same_genre_weight` as the share given to genre. Subtracting it from `1` gives the share left for semantic similarity. The two shares add to `1`, so their weighted average also stays between `0` and `1`.

For example, with `semantic_score = 0.8`, `genre_match = 0.4`, and `same_genre_weight = 0.15`, genre gets 15% and semantic similarity gets `1 - 0.15 = 0.85` (85%). The final score is `0.85 × 0.8 + 0.15 × 0.4 = 0.74`. The `0.15` weight is an illustrative choice, not one learned from listener data.

**How are songs sorted?** `ranked.append((score, distance, song))` saves one tuple for each unheard song. In the sort expression, `item` means one of those tuples: for `(0.8, 0.6, song_B)`, `item[0]` is the score `0.8`, `item[1]` is the distance `0.6`, and `item[2]` is the `Song` object. Python checks the following values in order, moving to the next one only when the previous values tie:

1. `-item[0]` puts higher scores first. Python normally sorts smaller numbers first; negating the score reverses that order.
2. `item[1]` puts the smaller Chroma distance first when scores are equal.
3. `item[2].songId` orders songs by ID when both score and distance are equal, so ties have a consistent order.

| Song | Score | Distance | Order after sorting |
| --- | ---: | ---: | ---: |
| A | 0.70 | 0.40 | 3 |
| B | 0.80 | 0.60 | 1 |
| C | 0.70 | 0.20 | 2 |

B comes first because it has the highest score. C and A tie on score, so C comes before A because its distance is smaller.


In [4]:
class SongRecommender:
    def __init__(self, songs: list[Song], collection):
        self.songs_by_id = {song.songId: song for song in songs}
        self.collection = collection

    def get_recommendations(
        self,
        listening_history: ListeningHistory,
        n_recommendations: int = 3,
        same_genre_weight: float = 0.0,
    ) -> list[Song]:
        """Rank unheard songs using the listener vector and an optional genre match."""
        if n_recommendations < 0:
            raise ValueError("n_recommendations must be nonnegative")
        if not 0 <= same_genre_weight <= 1:
            raise ValueError("same_genre_weight must be between 0 and 1")
        if not listening_history.songs or n_recommendations == 0:
            return []

        # Repeated plays do not count as separate likes in this example.
        liked_songs_by_id = {song.songId: song for song in listening_history.songs}
        liked_songs = list(liked_songs_by_id.values())
        liked_ids = set(liked_songs_by_id.keys())
        unknown_ids = liked_ids - self.songs_by_id.keys()
        if unknown_ids:
            raise ValueError(f"Songs missing from the catalog: {sorted(unknown_ids)}")

        stored = self.collection.get(ids=list(liked_ids), include=["embeddings"])
        embeddings = stored["embeddings"]
        if embeddings is None or len(embeddings) != len(liked_ids):
            raise ValueError("A liked song is missing an embedding in the collection")

        listener_vector = [
            sum(float(value) for value in dimension) / len(embeddings)
            for dimension in zip(*embeddings)
        ]
        if not listener_vector:
            raise ValueError("Stored song embeddings are empty")

        # Query the whole 24-song catalog so genre reranking sees every candidate.
        results = self.collection.query(
            query_embeddings=[listener_vector],
            n_results=self.collection.count(),
            include=["distances"],
        )
        ranked = []
        for song_id, distance in zip(results["ids"][0], results["distances"][0]):
            if listening_history.has_listened_to(song_id):
                continue
            song = self.songs_by_id[song_id]
            # Chroma cosine distance runs from 0 (closest) to 2 (farthest).
            # Divide by 2 to get 0..1, then subtract from 1 so higher means closer.
            # For example, distance 0.4 becomes 1 - 0.4 / 2 = 0.8.
            semantic_score = 1 - distance / 2
            # Share of unique liked songs with this candidate's exact genre.
            genre_match = sum(
                liked.genre == song.genre for liked in liked_songs
            ) / len(liked_songs)
            # Genre gets same_genre_weight of the score; semantic similarity
            # gets the remainder, 1 - same_genre_weight. For example, 0.15
            # for genre leaves 0.85 for semantic similarity.
            score = (
                (1 - same_genre_weight) * semantic_score
                + same_genre_weight * genre_match
            )
            ranked.append((score, distance, song))

        # Each item is (score, distance, song). Check these in order:
        # -item[0]: higher scores first (Python sorts smaller values first).
        # item[1]: smaller distances first when scores tie.
        # item[2].songId: consistent order when both values tie.
        ranked.sort(key=lambda item: (-item[0], item[1], item[2].songId))
        # Take the top entries and return only their Song objects.
        return [song for _, _, song in ranked[:n_recommendations]]


## Results

Our listener likes **First Light**, **Begin Again**, and **Morning Resolve**. Their descriptions concern heartbreak, recovery, and starting over. The next cell prints the top three unheard songs using the centroid and cosine ranking, with the genre preference set to zero.


In [5]:
songs_by_id = {song.songId: song for song in songs}
listener = User(username="demo_listener")
for song_id in ("song_001", "song_002", "song_006"):
    listener.add_listen(songs_by_id[song_id])
recommender = SongRecommender(songs, collection)
recommendations = recommender.get_recommendations(
    listener.listening_history, same_genre_weight=0.0
)

print("Liked:")
for song in listener.listening_history.songs:
    print(f"  {song.title} ({song.genre})")
print("\nRecommended unheard songs:")
for rank, song in enumerate(recommendations, start=1):
    print(f"  {rank}. {song.title} ({song.genre}) — {song.songId}")
    print(f"     {song.description}")


Liked:
  First Light (Pop)
  Begin Again (Indie pop)
  Morning Resolve (Acoustic)

Recommended unheard songs:
  1. Last Letter (Folk) — song_008
     A bittersweet song about saying goodbye to a former partner and learning to let go.
  2. Beside You (Pop) — song_020
     A tender love song about trust and finding comfort in a lasting relationship.
  3. Empty Room (Pop) — song_007
     A slow ballad about loneliness and missing someone after a painful breakup.


## Analysis

**Why did these songs rank highly?** In the saved run, **Last Letter** ranks first and shares the themes of goodbye and letting go. **Empty Room** also concerns a breakup. **Beside You** is about comfort in a relationship, which is a looser textual connection. These are plausible readings of the descriptions, not a way to inspect exactly what the embedding model learned.

**Why include genre?** A song can have a similar description but belong to a different genre. As an optional preference, we count how many distinct liked songs have the candidate's exact genre, then divide by the number of distinct liked songs. This gives a fraction from `0` to `1`. Here the listener likes Pop, Indie pop, and Acoustic, so a Pop candidate gets `1/3`; a Folk candidate gets `0`. `Pop` and `Indie pop` are different labels.

With `same_genre_weight=0`, genre has no effect on the ranking. At `0.15`, the score is 85% rescaled cosine similarity and 15% genre match. The comparison below shows how favoring a familiar genre can change the order; it cannot tell us whether the listener would enjoy the song.


In [6]:
for weight in (0.0, 0.15):
    titles = [
        song.title
        for song in recommender.get_recommendations(
            listener.listening_history, same_genre_weight=weight
        )
    ]
    print(f"Genre weight {weight:.2f}: {', '.join(titles)}")


Genre weight 0.00: Last Letter, Beside You, Empty Room
Genre weight 0.15: Beside You, Empty Room, Last Letter


In this run, the Pop genre match moves **Beside You** and **Empty Room** ahead of **Last Letter**. That is a design choice layered on the centroid experiment. Beside You is a love song rather than a song about getting over a breakup, so the new order is not automatically better.


## Failure cases

**Text similarity is not musical preference.** The model reads titles and descriptions, not audio. It cannot hear tempo, vocals, instrumentation, or production style, and a play is not necessarily a like. Averaging also loses detail when someone enjoys several different styles: one point in vector space may sit between them.

Try a second listener who likes quieter songs. The code still returns three unheard songs even if the catalog does not contain three strong matches. You can replace these song IDs with others from [`songs.csv`](songs.csv) to test a different history.


In [7]:
second_listener = User(username="quiet_listener")
for song_id in ("song_004", "song_005", "song_014"):
    second_listener.add_listen(songs_by_id[song_id])

print("Liked:", ", ".join(song.title for song in second_listener.listening_history.songs))
for song in recommender.get_recommendations(
    second_listener.listening_history, same_genre_weight=0.0
):
    print(f"Suggested: {song.title} ({song.genre}) — {song.description}")


Liked: Rain on the Window, Evening Drizzle, Quiet Pages
Suggested: Breathe Slowly (Ambient) — A peaceful instrumental track with flowing sounds for meditation and relaxation.
Suggested: Golden Afternoon (Dance pop) — A cheerful summer song about sunshine and dancing at an outdoor festival.
Suggested: Summer Pulse (Dance pop) — An energetic dance song for a summer party.


In the saved run, **Breathe Slowly** fits the quiet songs, while the dance tracks are surprising. This is a useful failure case: the method can fill the requested three slots without knowing whether the later results are good enough. A real recommender might require a minimum score and return fewer songs.


## Conclusion

**Preference vectors work as a simple content-based baseline, but lose information.** The centroid retrieves some thematically related, unheard songs from this small catalog. It compresses several interests into one vector, relies on song text rather than sound, and has not been evaluated against real listener preferences. These results support a limited claim about text retrieval, not a claim that we have captured musical taste.


## Where we’re going next

This notebook ends with a ranked list of unheard songs. To build on it, we can use **retrieval-augmented generation (RAG)** to explain those recommendations. We would give a language model the liked songs and the retrieved songs' titles, genres, and descriptions, then ask it to explain why each recommendation fits. The catalog records provide the context for its answer.

The flow becomes: listening history → preference vector → Chroma search → ranked unheard songs → explanation based on their descriptions. We can then check whether each explanation points to evidence in the retrieved descriptions. The model still cannot tell from this text how a song sounds or whether the listener would enjoy it.
